# Resume 30M on Colab T4

Your Drive folder already has the right uploads, just **different names**:
- `colab_upload.zip` = code + TinyStories npy (not `prototype.zip`)
- `checkpoints.zip` = `checkpoints/baseline-30m/last.pt` (not an unzipped folder)

Do **not** rename anything. Runtime → T4 GPU. Run cells 1 → 2 → 3. Cell 2 must print **READY**.

In [ ]:
!nvidia-smi -L
import torch
from pathlib import Path
from google.colab import drive

assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU, then Restart session."
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
else:
    print("Drive already mounted — skipping remount")
print(torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import shutil, zipfile, os, sys

DRIVE = Path("/content/drive/MyDrive/ttt-prototype")
code_zip = DRIVE / "colab_upload.zip"
ckpt_zip = DRIVE / "checkpoints.zip"

print("In Drive:", sorted(p.name for p in DRIVE.iterdir()) if DRIVE.exists() else "MISSING")
assert code_zip.exists(), f"Need {code_zip}"
assert ckpt_zip.exists(), f"Need {ckpt_zip}"

# Code + train.npy live inside colab_upload.zip as colab_upload/prototype/...
extract_code = Path("/content/from_drive")
if extract_code.exists():
    shutil.rmtree(extract_code)
extract_code.mkdir()
print("Unzipping code zip...")
with zipfile.ZipFile(code_zip) as zf:
    zf.extractall(extract_code)
matches = [p for p in extract_code.rglob("scripts/train.py") if "__MACOSX" not in str(p)]
assert matches, "colab_upload.zip has no scripts/train.py"
os.chdir(matches[0].parent.parent)
sys.path.insert(0, str(Path.cwd()))
print("working dir", Path.cwd())

# last.pt lives inside checkpoints.zip as checkpoints/baseline-30m/last.pt
extract_ckpt = Path("/content/from_ckpt_zip")
if extract_ckpt.exists():
    shutil.rmtree(extract_ckpt)
extract_ckpt.mkdir()
print("Unzipping checkpoints.zip (a few minutes)...")
with zipfile.ZipFile(ckpt_zip) as zf:
    zf.extractall(extract_ckpt)
found = [p for p in extract_ckpt.rglob("last.pt") if "baseline-30m" in str(p)]
assert found, "checkpoints.zip has no baseline-30m/last.pt"
src_ckpt = max(found, key=lambda p: p.stat().st_size)
print("using checkpoint", src_ckpt, "MB", round(src_ckpt.stat().st_size / 1e6, 1))
assert src_ckpt.stat().st_size > 300_000_000, "That last.pt is too small (old 17M). Need the ~376MB 30M file."

out = DRIVE / "checkpoints" / "baseline-30m"
out.mkdir(parents=True, exist_ok=True)
dst_ckpt = out / "last.pt"
if not dst_ckpt.exists() or dst_ckpt.stat().st_size != src_ckpt.stat().st_size:
    shutil.copy2(src_ckpt, dst_ckpt)

train_npy = Path("data/tinystories-v2/train.npy")
val_npy = Path("data/tinystories-v2/validation.npy")
tok = Path("data/tokenizer/tokenizer.json")
assert train_npy.exists() and val_npy.exists() and tok.exists(), (
    f"Data missing. train={train_npy.exists()} val={val_npy.exists()} tok={tok.exists()}"
)
print("READY")
print("train.npy MB", round(train_npy.stat().st_size / 1e6, 1))
print("last.pt MB", round(dst_ckpt.stat().st_size / 1e6, 1))

In [ ]:
%pip install -q tokenizers numpy
from pathlib import Path

out = Path("/content/drive/MyDrive/ttt-prototype/checkpoints/baseline-30m")
resume = out / "last.pt"
assert Path("scripts/train.py").exists(), "Run the READY cell first."
assert resume.exists() and resume.stat().st_size > 300_000_000

!python -u scripts/train.py --epochs 10 --optimizer adamw --weight-decay 0.1 \
  --device cuda --amp auto --seq-len 256 --batch-size 2 --grad-accum 2 \
  --data data/tinystories-v2/train.npy --val-data data/tinystories-v2/validation.npy \
  --out {out} --resume {resume}